In [ ]:
# Tabel perbandingan final dengan 4 baseline
comparison_final = pd.DataFrame([summary_qualy, summary_grid, summary_standings, summary_wf])

print('=' * 85)
print('PERBANDINGAN SEMUA BASELINE (2014-2025)')
print('=' * 85)
display(comparison_final[['label', 'podium_hit_rate', 'winner_accuracy', 'exact_drivers', 'exact_ordered', 'total_races']])

# Tabel dengan NDCG
print()
print('=' * 85)
print('NDCG@3 PER BASELINE')
print('=' * 85)
ndcg_data = pd.DataFrame({
    'Baseline': ['Qualifying Top-3', 'Grid Top-3', 'Standings Top-3', 'Weighted Formula'],
    'NDCG@3': [ndcg_qualy, ndcg_grid, ndcg_standings, ndcg_wf]
})
display(ndcg_data)

# Update hasil export dengan Weighted Formula
comparison_final.to_csv('../reports/metrics/baseline_results.csv', index=False)
print('\nBaseline results saved to reports/metrics/baseline_results.csv')

## Perbandingan Semua Baseline

| Metrik | Qualifying | Grid | Standings | Weighted Formula |
|---|---|---|---|---|

In [ ]:
# Evaluasi Weighted Formula sebagai baseline
# Gunakan wf_score sebagai dasar ranking: semakin tinggi score, semakin baik prediksi podium

baseline_wf = evaluate_baseline_per_race(df_modern_wf.dropna(subset=['wf_score']), 'wf_score', ascending=False)
summary_wf = print_baseline_summary(baseline_wf, 'Baseline Weighted Formula')

# NDCG@3 untuk Weighted Formula
ndcg_wf = calculate_ndcg_baseline(df_modern_wf.dropna(subset=['wf_score']), 'wf_score', ascending=False)
print(f'NDCG@3 - Weighted Formula: {ndcg_wf:.4f}')

In [ ]:
# Hitung Weighted Formula score per driver per race
# Normalisasi min-max per race agar setiap fitur berada di [0,1] (0 = terbaik)

def calculate_weighted_formula_score(df_data):
    """
    Hitung weighted formula score untuk setiap driver dalam setiap race.
    Score lebih rendah = lebih baik (mirip posisi).
    """
    df_data = df_data.copy()
    df_data['wf_score'] = np.nan
    
    for race_id, group in df_data.groupby('raceId'):
        group = group.copy()
        
        # Skip race dengan data tidak lengkap
        if group[available_features].isna().any().any():
            continue
        
        # Normalisasi min-max per race (0 = terbaik, 1 = terburuk)
        for feat in available_features:
            min_val = group[feat].min()
            max_val = group[feat].max()
            if max_val > min_val:
                group[f'{feat}_norm'] = (group[feat] - min_val) / (max_val - min_val)
            else:
                group[f'{feat}_norm'] = 0.0
        
        # Calculate score
        group['wf_score'] = (
            -0.45 * group['qualy_position_norm']
            -0.20 * group['grid_effective_norm']
            +0.15 * (1 - group['driver_finish_avg_5_norm'])  # finish position: lower = better, so invert
            +0.15 * (1 - group['team_points_avg_5_norm'])    # points: higher = better, so invert
            +0.05 * (1 - group['driver_circuit_avg_finish_norm'])  # lower = better, so invert
        )
        
        df_data.loc[group.index, 'wf_score'] = group['wf_score']
    
    return df_data

# Hitung score untuk data modern (>= 2014)
df_modern_wf = df_full[df_full['year'] >= 2014].copy() if 'year' in df_full.columns else df_full.copy()
df_modern_wf = calculate_weighted_formula_score(df_modern_wf)

print(f'Data dengan WF score: {df_modern_wf["wf_score"].notna().sum()} rows')
print(f'Missing WF score: {df_modern_wf["wf_score"].isna().sum()} rows')

In [ ]:
# Fitur yang dibutuhkan untuk Weighted Formula
weight_features = [
    'qualy_position', 'grid_effective',
    'driver_finish_avg_5', 'team_points_avg_5',
    'driver_circuit_avg_finish'
]

# Cek ketersediaan kolom
available_features = [f for f in weight_features if f in df_full.columns]
missing_features = [f for f in weight_features if f not in df_full.columns]

print(f'Fitur tersedia: {available_features}')
print(f'Fitur tidak tersedia: {missing_features}')

if missing_features:
    print('\nBeberapa fitur tidak tersedia. Weighted Formula tidak dapat dihitung.')
    print('Jalankan notebook 04_feature_engineering.ipynb untuk generate dataset lengkap.')
else:
    print('\nSemua fitur tersedia. Melanjutkan ke scoring...')

In [ ]:
# Weighted Formula Baseline
# Score = -0.45*qual_position - 0.20*grid + 0.15*driver_form + 0.15*constructor_form + 0.05*circuit_history

# Load parquet dengan fitur lengkap
try:
    df_full = pd.read_parquet('../data/processed/model_dataset.parquet')
    print(f'Loaded parquet: {df_full.shape}')
except:
    print('Parquet belum tersedia. Gunakan data dari merge sebelumnya.')
    print('Jalankan notebook 04_feature_engineering.ipynb terlebih dahulu untuk generate parquet.')
    # Fallback: merge dari data yang sudah ada di notebook ini
    df_full = df_modern.copy()
    # Merge constructor info
    constructors_info = pd.read_csv(DATA_PATH + 'constructors.csv')
    df_full = df_full.merge(constructors_info[['constructorId', 'name']], on='constructorId', how='left', suffixes=('', '_cons'))
    
print(f'Data: {df_full.shape}')
print(f'Tahun: {df_full["year"].min()} - {df_full["year"].max()}')

# Baseline Prediction - F1 Podium

Tujuan: Menghitung metrik baseline sederhana sebagai pembanding model ML.

## Baseline:
1. **Qualifying Top-3**: P1 = qualy position 1, P2 = qualy position 2, P3 = qualy position 3
2. **Grid Top-3**: P1 = grid 1, P2 = grid 2, P3 = grid 3
3. **Driver Standings Top-3**: P1, P2, P3 = posisi klasemen sebelum race

## Metrik per Race:
- Podium Hit Rate (jumlah podium aktual yang masuk 3 prediksi teratas)
- Exact Podium Drivers (3 pembalap podium benar, tanpa peduli urutan)
- Exact Ordered Podium (P1, P2, P3 tepat semua)
- Winner Accuracy (P1 benar)
- NDCG@3

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

DATA_PATH = '../data/'

In [ ]:
# Load data
results = pd.read_csv(DATA_PATH + 'results.csv', low_memory=False)
races = pd.read_csv(DATA_PATH + 'races.csv', low_memory=False)
qualifying = pd.read_csv(DATA_PATH + 'qualifying.csv', low_memory=False)
driver_standings = pd.read_csv(DATA_PATH + 'driver_standings.csv', low_memory=False)
drivers = pd.read_csv(DATA_PATH + 'drivers.csv', low_memory=False)

# Konversi tipe
results['grid'] = pd.to_numeric(results['grid'].replace(r'\N', np.nan, regex=False), errors='coerce')
results['positionOrder'] = pd.to_numeric(results['positionOrder'], errors='coerce')

print(f'Results: {results.shape}')
print(f'Races: {races.shape}')
print(f'Qualifying: {qualifying.shape}')
print(f'Driver Standings: {driver_standings.shape}')
print(f'Drivers: {drivers.shape}')

## Merge Data

Gabungkan results dengan races (untuk year, round), drivers (nama), dan qualifying.

In [ ]:
# Merge results + races
df = results.merge(races[['raceId', 'year', 'round', 'name', 'date']], on='raceId', how='left')
df = df.merge(drivers[['driverId', 'driverRef', 'code', 'forename', 'surname']], on='driverId', how='left')

# Urutkan berdasarkan tanggal
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date', 'raceId', 'positionOrder']).reset_index(drop=True)

print(f'Merged shape: {df.shape}')
print(f'Tahun: {df["year"].min()} - {df["year"].max()}')

In [ ]:
# Konversi posisi qualifying ke numerik
qualifying['position'] = pd.to_numeric(qualifying['position'].replace(r'\N', np.nan, regex=False), errors='coerce')
qualifying = qualifying.rename(columns={'position': 'qualy_position'})

# Merge qualifying
df = df.merge(qualifying[['raceId', 'driverId', 'qualy_position']], on=['raceId', 'driverId'], how='left')

print(f'Dengan qualifying: {df.shape}')
df[['year', 'round', 'driverId', 'qualy_position', 'grid', 'positionOrder']].head()

## Baseline 1: Qualifying Top-3

Untuk setiap race, ambil 3 pembalap dengan posisi qualifying terbaik (qualy_position = 1, 2, 3).

In [ ]:
# Filter data dari 2014 ke atas (era hybrid)
df_modern = df[df['year'] >= 2014].copy()

# Khusus data yang punya qualifying position
df_qualy = df_modern.dropna(subset=['qualy_position']).copy()
df_qualy['qualy_position'] = df_qualy['qualy_position'].astype(int)

print(f'Data modern (>=2014): {df_modern.shape}')
print(f'Data dengan qualifying: {df_qualy.shape}')

In [ ]:
# Fungsi untuk evaluasi baseline per race
def evaluate_baseline_per_race(df_data, position_col, ascending=True, n_top=3):
    """
    Evaluasi baseline: n_top pembalap berdasarkan position_col sebagai prediksi podium.
    position_col: kolom yang digunakan untuk menentukan urutan (qualy_position, grid, dll)
    ascending: True = posisi kecil lebih baik (qualifying), False = poin besar lebih baik (standings)
    """
    results_list = []
    
    for race_id, group in df_data.groupby('raceId'):
        # Aktual podium
        actual_podium = group[group['positionOrder'] <= 3]['driverId'].tolist()
        if len(actual_podium) < 3:
            continue
        
        # Aktual P1, P2, P3
        actual_p1 = group[group['positionOrder'] == 1]['driverId'].values
        actual_p2 = group[group['positionOrder'] == 2]['driverId'].values
        actual_p3 = group[group['positionOrder'] == 3]['driverId'].values
        
        if len(actual_p1) == 0 or len(actual_p2) == 0 or len(actual_p3) == 0:
            continue
        actual_p1 = actual_p1[0]
        actual_p2 = actual_p2[0]
        actual_p3 = actual_p3[0]
        
        # Prediksi: urutkan berdasarkan position_col, ambil n_top teratas
        predicted = group.dropna(subset=[position_col]).sort_values(position_col, ascending=ascending)
        predicted_top3 = predicted.head(n_top)['driverId'].tolist()
        predicted_p1 = predicted.head(1)['driverId'].values[0] if len(predicted) >= 1 else None
        predicted_p2 = predicted.iloc[1:2]['driverId'].values[0] if len(predicted) >= 2 else None
        predicted_p3 = predicted.iloc[2:3]['driverId'].values[0] if len(predicted) >= 3 else None
        
        # Metrik
        hits = sum(1 for d in actual_podium if d in predicted_top3)
        exact_drivers = all(d in predicted_top3 for d in actual_podium)
        exact_ordered = (actual_p1 == predicted_p1 and 
                         actual_p2 == predicted_p2 and 
                         actual_p3 == predicted_p3)
        winner_correct = actual_p1 == predicted_p1
        
        year = group['year'].iloc[0]
        race_name = group['name'].iloc[0]
        
        results_list.append({
            'raceId': race_id,
            'year': year,
            'race_name': race_name,
            'podium_hits': hits,
            'exact_drivers': int(exact_drivers),
            'exact_ordered': int(exact_ordered),
            'winner_correct': int(winner_correct),
            'n_predicted': len(predicted_top3)
        })
    
    return pd.DataFrame(results_list)


# Hitung baseline qualifying
baseline_qualy = evaluate_baseline_per_race(df_qualy, 'qualy_position', ascending=True)
print(f'Baseline Qualifying: {len(baseline_qualy)} race')
baseline_qualy.head()

In [ ]:
# Ringkasan metrik baseline qualifying
def print_baseline_summary(baseline_df, label):
    total = len(baseline_df)
    total_hits = baseline_df['podium_hits'].sum()
    possible_hits = total * 3
    
    print(f'=== {label} ===')
    print(f'Total race: {total}')
    print(f'Podium Hit Rate: {total_hits}/{possible_hits} = {total_hits/possible_hits:.4f}')
    print(f'Rata-rata Podium Hits per race: {baseline_df["podium_hits"].mean():.4f}')
    print(f'Exact Podium Drivers: {baseline_df["exact_drivers"].mean():.4f} ({baseline_df["exact_drivers"].sum()}/{total})')
    print(f'Exact Ordered Podium: {baseline_df["exact_ordered"].mean():.4f} ({baseline_df["exact_ordered"].sum()}/{total})')
    print(f'Winner Accuracy: {baseline_df["winner_correct"].mean():.4f} ({baseline_df["winner_correct"].sum()}/{total})')
    print()
    return {
        'label': label,
        'total_races': total,
        'podium_hit_rate': total_hits / possible_hits,
        'avg_podium_hits': baseline_df['podium_hits'].mean(),
        'exact_drivers': baseline_df['exact_drivers'].mean(),
        'exact_ordered': baseline_df['exact_ordered'].mean(),
        'winner_accuracy': baseline_df['winner_correct'].mean()
    }

summary_qualy = print_baseline_summary(baseline_qualy, 'Baseline Qualifying Top-3')

## Baseline 2: Grid Top-3

Prediksi P1 = grid 1, P2 = grid 2, P3 = grid 3.

In [ ]:
# Koreksi grid 0 (pit lane start) -> treat as grid terakhir
df_modern['grid_effective'] = df_modern['grid'].copy()

# Cari field_size per race
field_size = df_modern.groupby('raceId')['grid'].max()
df_modern['field_size'] = df_modern['raceId'].map(field_size)

# Grid 0 = pit lane start, set ke field_size + 1
mask_grid0 = df_modern['grid_effective'] == 0
df_modern.loc[mask_grid0, 'grid_effective'] = df_modern.loc[mask_grid0, 'field_size'] + 1

# Baseline grid
baseline_grid = evaluate_baseline_per_race(df_modern, 'grid_effective', ascending=True)
summary_grid = print_baseline_summary(baseline_grid, 'Baseline Grid Top-3')

## Baseline 3: Driver Standings Top-3

Prediksi P1, P2, P3 = posisi klasemen tertinggi SEBELUM race.

**Peringatan Leakage:** Kolom `position` di `driver_standings.csv` menyimpan posisi SETELAH race. Kita harus melakukan shift(1) per musim.

In [ ]:
# Load races untuk mapping year-round
races_info = races[['raceId', 'year', 'round']].copy()
races_info['date'] = pd.to_datetime(races['date'])

# Merge standings dengan info race
standings = driver_standings.copy()
standings = standings.merge(races_info, on='raceId', how='left')
standings['position'] = pd.to_numeric(standings['position'], errors='coerce')
standings['points'] = pd.to_numeric(standings['points'], errors='coerce')

# Sort per driver agar bisa shift
standings = standings.sort_values(['driverId', 'year', 'round']).reset_index(drop=True)

# Shift: posisi sebelum race = posisi di race sebelumnya
standings['prev_position'] = standings.groupby('driverId')['position'].shift(1)
standings['prev_points'] = standings.groupby('driverId')['points'].shift(1)

# Untuk race pertama musim, ambil posisi akhir musim sebelumnya
# Shift dalam satu driverId sudah menangani ini, tapi untuk race pertama overall hasilnya NaN
# Kita isi dengan posisi standar (20 = posisi terendah asumsi)
standings['prev_position'] = standings['prev_position'].fillna(20)
standings['prev_points'] = standings['prev_points'].fillna(0)

standings[['raceId', 'year', 'round', 'driverId', 'position', 'prev_position', 'prev_points']].head(10)

In [ ]:
# Merge prev_position ke dataframe utama
df_modern = df_modern.merge(
    standings[['raceId', 'driverId', 'prev_position', 'prev_points']],
    on=['raceId', 'driverId'],
    how='left'
)

# Baseline standings (prev_position lebih kecil = lebih baik)
baseline_standings = evaluate_baseline_per_race(df_modern, 'prev_position', ascending=True)
summary_standings = print_baseline_summary(baseline_standings, 'Baseline Standings Top-3')

## Perbandingan Baseline

| Metrik | Qualifying Top-3 | Grid Top-3 | Standings Top-3 |
|---|---|---|---|

In [ ]:
# Tabel perbandingan
comparison = pd.DataFrame([summary_qualy, summary_grid, summary_standings])
display(comparison[['label', 'podium_hit_rate', 'winner_accuracy', 'exact_drivers', 'exact_ordered', 'total_races']])

## Baseline per Musim

Lihat konsistensi baseline antar musim.

In [ ]:
def baseline_per_season(baseline_df):
    season_stats = baseline_df.groupby('year').agg(
        total_races=('raceId', 'count'),
        podium_hit_rate=('podium_hits', lambda x: x.sum() / (len(x) * 3)),
        winner_accuracy=('winner_correct', 'mean'),
        exact_drivers=('exact_drivers', 'mean'),
        exact_ordered=('exact_ordered', 'mean')
    ).reset_index()
    return season_stats

print('--- Baseline Qualifying per Season ---')
qualy_season = baseline_per_season(baseline_qualy)
display(qualy_season)

print('--- Baseline Grid per Season ---')
grid_season = baseline_per_season(baseline_grid)
display(grid_season)

In [ ]:
# Simpan hasil baseline
comparison.to_csv('../reports/metrics/baseline_results.csv', index=False)
print('Baseline results saved to reports/metrics/baseline_results.csv')

## NDCG@3

Hitung NDCG@3 untuk setiap baseline.

In [ ]:
def ndcg_at_k(relevance_scores, k=3):
    """
    Hitung NDCG@k.
    relevance: P1=3, P2=2, P3=1, non-podium=0
    """
    dcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(relevance_scores[:k]))
    
    # Ideal DCG: urutkan relevance descending
    ideal = sorted(relevance_scores, reverse=True)[:k]
    idcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal))
    
    return dcg / idcg if idcg > 0 else 0


def calculate_ndcg_baseline(df_data, position_col, ascending=True):
    """Hitung NDCG@3 untuk baseline berdasarkan position_col"""
    ndcg_scores = []
    
    for race_id, group in df_data.groupby('raceId'):
        # Relevance scores aktual
        group = group.copy()
        group['relevance'] = group['positionOrder'].apply(
            lambda x: 3 if x == 1 else (2 if x == 2 else (1 if x == 3 else 0))
        )
        
        # Urutkan berdasarkan position_col
        predicted = group.dropna(subset=[position_col]).sort_values(position_col, ascending=ascending)
        
        if len(predicted) < 3:
            continue
        
        predicted_relevance = predicted.head(3)['relevance'].tolist()
        ndcg = ndcg_at_k(predicted_relevance, k=3)
        ndcg_scores.append(ndcg)
    
    return np.mean(ndcg_scores)


ndcg_qualy = calculate_ndcg_baseline(df_qualy, 'qualy_position', ascending=True)
ndcg_grid = calculate_ndcg_baseline(df_modern, 'grid_effective', ascending=True)
ndcg_standings = calculate_ndcg_baseline(df_modern, 'prev_position', ascending=True)

print(f'NDCG@3 - Qualifying: {ndcg_qualy:.4f}')
print(f'NDCG@3 - Grid: {ndcg_grid:.4f}')
print(f'NDCG@3 - Standings: {ndcg_standings:.4f}')

In [ ]:
# Final summary dengan NDCG
print('=' * 65)
print('SUMMARY BASELINE (2014-2025)')
print('=' * 65)
print(f'{"Metrik":<25} {"Qualifying":<15} {"Grid":<15} {"Standings":<15}')
print('-' * 65)
print(f'{"Podium Hit Rate":<25} {summary_qualy["podium_hit_rate"]:<15.4f} {summary_grid["podium_hit_rate"]:<15.4f} {summary_standings["podium_hit_rate"]:<15.4f}')
print(f'{"Winner Accuracy":<25} {summary_qualy["winner_accuracy"]:<15.4f} {summary_grid["winner_accuracy"]:<15.4f} {summary_standings["winner_accuracy"]:<15.4f}')
print(f'{"Exact Drivers":<25} {summary_qualy["exact_drivers"]:<15.4f} {summary_grid["exact_drivers"]:<15.4f} {summary_standings["exact_drivers"]:<15.4f}')
print(f'{"Exact Ordered":<25} {summary_qualy["exact_ordered"]:<15.4f} {summary_grid["exact_ordered"]:<15.4f} {summary_standings["exact_ordered"]:<15.4f}')
print(f'{"NDCG@3":<25} {ndcg_qualy:<15.4f} {ndcg_grid:<15.4f} {ndcg_standings:<15.4f}')
print('-' * 65)
print(f'Total races: {summary_qualy["total_races"]}')

## Baseline 4: Weighted Formula

Menggabungkan qualifying, driver form, constructor form, dan circuit history dengan bobot tetap.

Formula:
```
score = -0.45 * normalized_qual_position
        -0.20 * normalized_grid
        +0.15 * normalized_driver_form
        +0.15 * normalized_constructor_form
        +0.05 * normalized_circuit_history
```